In [ ]:
!pip install streamlit -q
!npm install -g localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 29.1 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [ ]:
# 1. SETUP & IMPORTS
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# --- DATA LOADING ---
def load_data():
    files = {
        "inst": 'Institution.csv',
        "revs": 'Reviewer.csv',
        "prof": 'Reviewer profile.csv',
        "cal": 'Reviewer calendar.csv',
        "past": 'Inspection Visit.csv',
        "reqs": 'visits requirements.csv'
    }
    data = {}
    try:
        for key, path in files.items():
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            data[key] = df
        return data
    except FileNotFoundError as e:
        print(f"❌ Error: Missing file - {e.filename}")
        return None

db = load_data()

# --- UI WIDGETS ---
if db:
    style = {'description_width': '120px'}
    layout = widgets.Layout(width='400px')
    title_html = widgets.HTML("<h2 style='text-align:center;'>🏛️ MOHESR: Unplanned Visit Matcher</h2>")

    inst_selector = widgets.Dropdown(options=sorted(db["inst"]["Institute ID"].unique()), description='Institute ID:', style=style, layout=layout)
    type_selector = widgets.Dropdown(options=sorted(db["reqs"]["Visit Type"].unique()), description='Visit Type:', style=style, layout=layout)
    loc_selector = widgets.Dropdown(options=sorted(db["inst"]["Location"].unique()), description='Location:', style=style, layout=layout)
    date_picker = widgets.DatePicker(description='Start Date:', value=datetime(2026, 9, 1), style=style, layout=layout)

    btn_recommend = widgets.Button(description='🔍 Recommend Best Match', button_style='primary', layout=widgets.Layout(width='300px', height='40px'))
    output_area = widgets.Output()

    # --- LOGIC FUNCTION ---
    def on_click_recommend(b):
        output_area.clear_output()
        with output_area:
            # 1. Get Inputs & Rules
            i_id = inst_selector.value
            v_type = type_selector.value
            loc = loc_selector.value

            rule = db["reqs"][db["reqs"]['Visit Type'] == v_type].iloc[0]
            needed = int(rule['Required Members'])
            lead_req = str(rule['Leader Required']).strip().lower() == 'yes'
            k_col = 'Required Knowledge' if 'Required Knowledge' in rule else 'Required Knowledge '
            skill = str(rule[k_col]).strip().lower()
            inst_name = db["inst"][db["inst"]["Institute ID"] == i_id]["Institute Name"].iloc[0]

            eligible = []
            fails = {"Skill/Expertise": 0, "Location": 0, "Past Visit": 0, "COI/Alumni": 0, "Vacation": 0}

            # 2. Filtering
            for _, r in db["revs"].iterrows():
                rid = r['Reviewer ID']
                # Expertise check
                if skill not in str(r['Knowledge Area']).lower():
                    fails["Skill/Expertise"] += 1; continue
                # Location check
                if r['Location'] != loc:
                    fails["Location"] += 1; continue
                # Past Visit check
                past_rec = db["past"][db["past"]['Institute ID'] == i_id]
                if not past_rec.empty and any(str(rid) in str(x) for x in past_rec['Assigned Reviewers'].values):
                    fails["Past Visit"] += 1; continue
                # COI/Alumni
                p_data = db["prof"][db["prof"]['Reviewer ID'] == rid]
                is_alumni = not p_data.empty and inst_name.lower() in str(p_data.iloc[0]['University of study']).lower()
                if str(r['Conflict Institution ID']) == str(i_id) or is_alumni:
                    fails["COI/Alumni"] += 1; continue

                eligible.append(r)

            # 3. Role Logic
            final_team = []
            df_pool = pd.DataFrame(eligible)
            if not df_pool.empty:
                df_pool = df_pool.sort_values("Performance Score", ascending=False)
                if v_type == "Renewal Visits":
                    members_only = df_pool[~df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    final_team = members_only.iloc[:needed].to_dict('records')
                elif lead_req:
                    leaders = df_pool[df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    members = df_pool[~df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    if not leaders.empty:
                        final_team.append(leaders.iloc[0].to_dict())
                        final_team.extend(members.iloc[:needed-1].to_dict('records'))
                else:
                    final_team = df_pool.iloc[:needed].to_dict('records')

            # 4. Display Results with Recommendation Logic
            clear_output()
            if len(final_team) == needed:
                display(widgets.HTML(f"<h3 style='color:green; text-align:center'>✅ Success! Team Found</h3>"))
                display(pd.DataFrame(final_team)[['Reviewer ID', 'Reviewer Name', 'Default Role', 'Performance Score']])
            else:
                display(widgets.HTML(f"<h3 style='color:red; text-align:center'>⚠️ Match Incomplete ({len(final_team)}/{needed})</h3>"))

                # Dynamic Justification Text
                just_text = "Disqualified candidates: " + ", ".join([f"{v} due to {k}" for k, v in fails.items() if v > 0])
                print(f"JUSTIFICATION: {just_text}")

                # --- STRATEGIC RECOMMENDATION ---
                print("-" * 30)
                print("💡 RECOMMENDATION:")
                if fails["Location"] > 0 and fails["Skill/Expertise"] == 0:
                    print(f"👉 Priority: Reallocate '{skill}' experts from other locations to {loc} for this visit.")
                elif fails["Skill/Expertise"] > 0 and len(final_team) == 0:
                    print(f"👉 Priority: No inspectors found with '{skill}' expertise. Consider using a Generalist or an external subject matter expert.")
                elif fails["Past Visit"] > 0 or fails["COI/Alumni"] > 0:
                    print("👉 Priority: Conflicts found with local staff. You must use out-of-city inspectors to ensure neutrality.")
                elif fails["Vacation"] > 0:
                    print("👉 Priority: Critical staff on leave. Shift visit date by +/- 7 days to clear vacation blocks.")
                else:
                    print("👉 Priority: Broaden search criteria or manually override the 'Renewal Visit' role restriction.")

    btn_recommend.on_click(on_click_recommend)
    button_box = widgets.HBox([btn_recommend], layout=widgets.Layout(display='flex', justify_content='center', margin='20px 0px'))
    display(widgets.VBox([title_html, widgets.HBox([inst_selector, type_selector], layout=widgets.Layout(justify_content='center')),
                          widgets.HBox([loc_selector, date_picker], layout=widgets.Layout(justify_content='center')),
                          button_box, output_area]))

INCLUDE THE ANNUAL PLAN
bold text

In [ ]:
# 1. SETUP & IMPORTS
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta

# --- DATA LOADING ---
def load_data():
    files = {
        "inst": 'Institution.csv',
        "revs": 'Reviewer.csv',
        "prof": 'Reviewer profile.csv',
        "cal": 'Reviewer calendar.csv',
        "past": 'Inspection Visit.csv',
        "reqs": 'visits requirements.csv',
        "annual": 'Annual_MOHESR_Plan.csv' # Added Annual Plan as input
    }
    data = {}
    try:
        for key, path in files.items():
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            data[key] = df
        return data
    except FileNotFoundError as e:
        print(f"❌ Error: Missing file - {e.filename}. Ensure you ran the annual plan script first.")
        return None

db = load_data()

# --- UI WIDGETS ---
if db:
    style = {'description_width': '120px'}
    layout = widgets.Layout(width='400px')
    title_html = widgets.HTML("<h2 style='text-align:center;'>🏛️ MOHESR: Unplanned Visit Matcher</h2>")

    inst_selector = widgets.Dropdown(options=sorted(db["inst"]["Institute ID"].unique()), description='Institute ID:', style=style, layout=layout)
    type_selector = widgets.Dropdown(options=sorted(db["reqs"]["Visit Type"].unique()), description='Visit Type:', style=style, layout=layout)
    loc_selector = widgets.Dropdown(options=sorted(db["inst"]["Location"].unique()), description='Location:', style=style, layout=layout)
    date_picker = widgets.DatePicker(description='Start Date:', value=datetime(2026, 9, 1), style=style, layout=layout)

    btn_recommend = widgets.Button(description='🔍 Recommend Best Match', button_style='primary', layout=widgets.Layout(width='300px', height='40px'))
    output_area = widgets.Output()

    # --- LOGIC FUNCTION ---
    def on_click_recommend(b):
        output_area.clear_output()
        with output_area:
            # 1. Setup Request Parameters
            i_id = inst_selector.value
            v_type = type_selector.value
            loc = loc_selector.value
            v_start = pd.to_datetime(date_picker.value)

            rule = db["reqs"][db["reqs"]['Visit Type'] == v_type].iloc[0]
            duration = int(rule['Duration (Days)'])
            v_end = v_start + timedelta(days=duration - 1)

            needed = int(rule['Required Members'])
            lead_req = str(rule['Leader Required']).strip().lower() == 'yes'
            k_col = 'Required Knowledge' if 'Required Knowledge' in rule else 'Required Knowledge '
            skill = str(rule[k_col]).strip().lower()
            inst_name = db["inst"][db["inst"]["Institute ID"] == i_id]["Institute Name"].iloc[0]

            eligible = []
            fails = {"Skill/Expertise": 0, "Location": 0, "Past Visit": 0, "COI/Alumni": 0, "Vacation": 0, "Annual Plan Busy": 0}

            # 2. Filtering
            for _, r in db["revs"].iterrows():
                rid = str(r['Reviewer ID'])

                # Expertise check
                if skill not in str(r['Knowledge Area']).lower():
                    fails["Skill/Expertise"] += 1; continue
                # Location check
                if r['Location'] != loc:
                    fails["Location"] += 1; continue
                # Past Visit check
                past_rec = db["past"][db["past"]['Institute ID'] == i_id]
                if not past_rec.empty and any(rid in str(x) for x in past_rec['Assigned Reviewers'].values):
                    fails["Past Visit"] += 1; continue
                # COI/Alumni
                p_data = db["prof"][db["prof"]['Reviewer ID'] == rid]
                is_alumni = not p_data.empty and inst_name.lower() in str(p_data.iloc[0]['University of study']).lower()
                if str(r['Conflict Institution ID']) == str(i_id) or is_alumni:
                    fails["COI/Alumni"] += 1; continue

                # --- NEW: Annual Plan Conflict Check ---
                conflict = False
                for _, plan_row in db["annual"].iterrows():
                    # Parse dates from string "YYYY-MM-DD to YYYY-MM-DD"
                    try:
                        p_dates = plan_row['Dates'].split(' to ')
                        ps, pe = pd.to_datetime(p_dates[0]), pd.to_datetime(p_dates[1])
                        assigned_revs = [x.strip() for x in str(plan_row['Assigned Reviewers']).split(',')]

                        # Check if reviewer is in this plan and if dates overlap
                        if rid in assigned_revs:
                            if not (v_end < ps or v_start > pe):
                                conflict = True
                                break
                    except: continue # Skip malformed plan lines

                if conflict:
                    fails["Annual Plan Busy"] += 1; continue

                eligible.append(r)

            # 3. Role Logic (1 Leader Strict / Renewal No Leader)
            final_team = []
            df_pool = pd.DataFrame(eligible)
            if not df_pool.empty:
                df_pool = df_pool.sort_values("Performance Score", ascending=False)
                if v_type == "Renewal Visits":
                    members_only = df_pool[~df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    final_team = members_only.iloc[:needed].to_dict('records')
                elif lead_req:
                    leaders = df_pool[df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    members = df_pool[~df_pool['Default Role'].str.lower().str.contains('leader', na=False)]
                    if not leaders.empty:
                        final_team.append(leaders.iloc[0].to_dict())
                        final_team.extend(members.iloc[:needed-1].to_dict('records'))
                else:
                    final_team = df_pool.iloc[:needed].to_dict('records')

            # 4. Results
            clear_output()
            if len(final_team) == needed:
                display(widgets.HTML(f"<h3 style='color:green; text-align:center'>✅ Success! Team Found (No Annual Plan Conflicts)</h3>"))
                display(pd.DataFrame(final_team)[['Reviewer ID', 'Reviewer Name', 'Default Role', 'Performance Score']])
            else:
                display(widgets.HTML(f"<h3 style='color:red; text-align:center'>⚠️ Match Incomplete ({len(final_team)}/{needed})</h3>"))

                # Explained Justification
                explanation = []
                for k, v in fails.items():
                    if v > 0:
                        if k == "Annual Plan Busy": explanation.append(f"{v} are already assigned in the Annual Plan for these dates")
                        else: explanation.append(f"{v} failed {k}")

                print(f"JUSTIFICATION: " + "; ".join(explanation))

                print("-" * 30)
                print("💡 RECOMMENDATION:")
                if fails["Annual Plan Busy"] > 0:
                    print("👉 Priority: Reviewer(s) are already booked for planned visits. Reschedule this unplanned visit or source external experts.")
                elif fails["Location"] > 0:
                    print(f"👉 Priority: Local pool exhausted. Reallocate '{skill}' experts from other cities.")
                else:
                    print("👉 Priority: Broaden search criteria.")

    btn_recommend.on_click(on_click_recommend)
    button_box = widgets.HBox([btn_recommend], layout=widgets.Layout(display='flex', justify_content='center', margin='20px 0px'))
    display(widgets.VBox([title_html, widgets.HBox([inst_selector, type_selector], layout=widgets.Layout(justify_content='center')),
                          widgets.HBox([loc_selector, date_picker], layout=widgets.Layout(justify_content='center')),
                          button_box, output_area]))

Automatic **Location**

In [ ]:
# 1. SETUP & IMPORTS
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta

# --- DATA LOADING ---
def load_data():
    files = {
        "inst": 'Institution.csv',
        "revs": 'Reviewer.csv',
        "prof": 'Reviewer profile.csv',
        "cal": 'Reviewer calendar.csv',
        "past": 'Inspection Visit.csv',
        "reqs": 'visits requirements.csv',
        "annual": 'Annual_MOHESR_Plan.csv'
    }
    data = {}
    try:
        for key, path in files.items():
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            if key == "cal":
                df['Vacation Start Date'] = pd.to_datetime(df['Vacation Start Date'])
                df['Vacation End Date'] = pd.to_datetime(df['Vacation End Date'])
            data[key] = df
        return data
    except Exception as e:
        display(widgets.HTML(f"<h3 style='color:red'>Data Error: {str(e)}</h3>"))
        return None

db = load_data()

# --- UI WIDGETS ---
if db:
    style = {'description_width': '120px'}
    layout = widgets.Layout(width='400px')
    title_html = widgets.HTML("<h2 style='text-align:center; color:#2c3e50'>🏛️ MOHESR: Unplanned Visit Matcher</h2>")

    inst_selector = widgets.Dropdown(options=sorted(db["inst"]["Institute ID"].unique()), description='Institute ID:', style=style, layout=layout)
    type_selector = widgets.Dropdown(options=sorted(db["reqs"]["Visit Type"].unique()), description='Visit Type:', style=style, layout=layout)
    date_picker = widgets.DatePicker(description='Start Date:', value=datetime(2026, 9, 1), style=style, layout=layout)
    btn_recommend = widgets.Button(description='🔍 Recommend Best Match', button_style='primary', layout=widgets.Layout(width='300px', height='40px'))
    output_area = widgets.Output()

    def on_click_recommend(b):
        output_area.clear_output()
        with output_area:
            # --- 1. SETUP PARAMETERS ---
            i_id = inst_selector.value
            v_type = type_selector.value
            v_start = pd.to_datetime(date_picker.value)

            inst_info = db["inst"][db["inst"]["Institute ID"] == i_id].iloc[0]
            loc = inst_info["Location"]
            inst_name = inst_info["Institute Name"]

            rule = db["reqs"][db["reqs"]['Visit Type'] == v_type].iloc[0]
            duration = int(rule['Duration (Days)'])
            v_end = v_start + timedelta(days=duration - 1)

            needed = int(rule['Required Members'])
            skill = str(rule.get('Required Knowledge', rule.get('Required Knowledge ', ''))).strip().lower()

            # Header Info
            display(widgets.HTML(f"<div style='border:1px solid #ddd;padding:10px;background:#f9f9f9;border-radius:5px;'><b>Target:</b> {inst_name} ({loc})<br><b>Period:</b> {v_start.date()} to {v_end.date()}</div>"))

            # --- 2. FILTERING ---
            eligible = []
            fails = {"Expertise": 0, "Location": 0, "Past Visit": 0, "COI/Alumni": 0, "Vacation": 0, "Annual Plan Busy": 0}

            total_reviewers = len(db["revs"])

            for _, r in db["revs"].iterrows():
                rid = str(r['Reviewer ID'])

                # Expertise check
                if skill not in str(r['Knowledge Area']).lower():
                    fails["Expertise"] += 1; continue

                # Location check
                if str(r['Location']).strip() != str(loc).strip():
                    fails["Location"] += 1; continue

                # Past Visit
                past_rec = db["past"][db["past"]['Institute ID'] == i_id]
                if not past_rec.empty and any(rid in str(x) for x in past_rec['Assigned Reviewers'].values):
                    fails["Past Visit"] += 1; continue

                # COI
                p_data = db["prof"][db["prof"]['Reviewer ID'].astype(str) == rid]
                is_alumni = not p_data.empty and inst_name.lower() in str(p_data.iloc[0]['University of study']).lower()
                if str(r['Conflict Institution ID']) == str(i_id) or is_alumni:
                    fails["COI/Alumni"] += 1; continue

                # Vacation
                rev_cal = db["cal"][db["cal"]['Reviewer ID'].astype(str) == rid]
                on_vacation = any(not (v_end < vs or v_start > ve) for vs, ve in zip(rev_cal['Vacation Start Date'], rev_cal['Vacation End Date']))
                if on_vacation:
                    fails["Vacation"] += 1; continue

                # Annual Plan
                conflict = False
                for _, plan_row in db["annual"].iterrows():
                    try:
                        p_dates = plan_row['Dates'].split(' to ')
                        ps, pe = pd.to_datetime(p_dates[0]), pd.to_datetime(p_dates[1])
                        assigned_revs = [x.strip() for x in str(plan_row['Assigned Reviewers']).split(',')]
                        if rid in assigned_revs and not (v_end < ps or v_start > pe):
                            conflict = True; break
                    except: continue
                if conflict:
                    fails["Annual Plan Busy"] += 1; continue

                eligible.append(r)

            # --- 3. FINAL DISPLAY ---
            final_df = pd.DataFrame(eligible)
            found_count = len(final_df)

            if found_count >= needed:
                display(widgets.HTML("<h3 style='color:green;text-align:center;'>✅ Team Successfully Matched</h3>"))
                display(final_df.head(needed)[['Reviewer ID', 'Reviewer Name', 'Default Role', 'Performance Score']])
            else:
                display(widgets.HTML(f"<h3 style='color:red;text-align:center;'>⚠️ Match Incomplete ({found_count}/{needed})</h3>"))
                if found_count > 0:
                    display(widgets.HTML("<b>Available Reviewers Found:</b>"))
                    display(final_df[['Reviewer ID', 'Reviewer Name', 'Default Role', 'Performance Score']])

                # --- FORCED JUSTIFICATION BLOCK ---
                # This block runs NO MATTER WHAT if the match is incomplete
                expl = []
                if fails["Expertise"] > 0: expl.append(f"<li><b>{fails['Expertise']}</b> lack '{skill}' specialization.</li>")
                if fails["Location"] > 0: expl.append(f"<li><b>{fails['Location']}</b> are located outside of {loc}.</li>")
                if fails["Annual Plan Busy"] > 0: expl.append(f"<li><b>{fails['Annual Plan Busy']}</b> are busy with the Annual Plan.</li>")
                if fails["Past Visit"] > 0: expl.append(f"<li><b>{fails['Past Visit']}</b> have previously visited this site.</li>")
                if fails["COI/Alumni"] > 0: expl.append(f"<li><b>{fails['COI/Alumni']}</b> have a Conflict of Interest.</li>")
                if fails["Vacation"] > 0: expl.append(f"<li><b>{fails['Vacation']}</b> are currently on leave.</li>")

                just_html = "".join(expl) if expl else "<li>No specific blockers identified; check if the database pool is empty.</li>"

                # Recommendation Logic
                rec = "Expand the search to other locations."
                if fails["Location"] > 10: rec = f"<b>Action:</b> Local pool in {loc} is exhausted. Please authorize travel for experts from other emirates."
                elif fails["Expertise"] > 10: rec = f"<b>Action:</b> Shortage of '{skill}' experts. Request external SME support."
                elif fails["Annual Plan Busy"] > 0: rec = "<b>Action:</b> Scheduling overlap. Shift visit by 1 week to free up Annual Plan reviewers."

                display(widgets.HTML(f"""
                <div style="background:#fff3cd; padding:15px; border-radius:8px; border-left:6px solid #ffc107; margin-top:10px; font-family:sans-serif;">
                    <h4 style="margin:0 0 10px 0; color:#856404;">📝 JUSTIFICATION</h4>
                    <ul style="padding-left:20px; margin:0;">{just_html}</ul>
                    <hr style="border:0; border-top:1px solid #ffeeba; margin:10px 0;">
                    <h4 style="margin:0 0 5px 0; color:#856404;">💡 RECOMMENDATION</h4>
                    <p style="margin:0;">{rec}</p>
                </div>
                """))

    btn_recommend.on_click(on_click_recommend)
    display(widgets.VBox([title_html, widgets.HBox([inst_selector, type_selector], layout=widgets.Layout(justify_content='center')), widgets.HBox([date_picker], layout=widgets.Layout(justify_content='center')), widgets.HBox([btn_recommend], layout=widgets.Layout(justify_content='center', margin='10px')), output_area]))